In [ ]:
import requests
import pandas as pd
import time

# --- Configuration ---
access_token = ''
headers = {'Authorization': f'token {access_token}'} if access_token else {}

prop_list = [
    'html_url', 'fork', 'created_at', 'updated_at', 'pushed_at', 'git_url',
    'size', 'stargazers_count', 'watchers_count', 'language', 'forks_count',
    'archived', 'disabled', 'open_issues_count', 'license', 'allow_forking'
]

csv_file_path = 'individual_repositories_detailed.csv'
output_csv_path = "final_repo_collection.csv"

# --- Filtering and Target Configuration ---
target_count = 10000
target_languages = {'Java', 'C', 'C++', 'JavaScript', 'Python', 'C#', 'TypeScript'}
BLACKLISTED_OWNERS = {
    'google', 'microsoft', 'facebook', 'apple', 'amazon', 'netflix', 'ibm',
    'oracle', 'intel', 'adobe', 'airbnb', 'uber', 'linkedin', 'twitter',
    'mozilla', 'apache', 'torvalds', 'docker', 'kubernetes', 'tensorflow',
    'pytorch', 'angular', 'vuejs', 'reactjs', 'nodejs', 'golang', 'rust-lang',
    'jetbrains', 'elastic', 'mongodb', 'automattic', 'square', 'shopify',
    'stripe', 'spotify', 'dropbox', 'github', 'gitlab', 'atlassian', 'slack',
    'zoom', 'salesforce', 'unity', 'unreal', 'epic', 'valve', 'steam',
    'alphabet', 'samsung', 'samsungelectronics', 'honhai', 'honhaiprecision',
    'meta', 'metaplatforms', 'huawei', 'huaweiinvestment', 'sony', 'dell',
    'delltechnologies', 'tencent', 'tencentholdings', 'taiwan', 'taiwansemiconductor',
    'tsmc', 'hitachi', 'lg', 'lgelectronics', 'accenture', 'nvidia', 'panasonic',
    'panasonicholdings', 'cisco', 'ciscosystems', 'lenovo', 'lenovogroup',
    'hp', 'pegatron', 'xiaomi', 'ubertech', 'ubertechnologies', 'qualcomm',
    'broadcom', 'chinaelectronics', 'quanta', 'quantacomputer', 'jabil',
    'sap', 'sapse', 'luxshare', 'luxshareprecision',
    'gnu', 'gnulinux', 'linuxfoundation', 'oniro', 'railcasts', 'cloudnative',
    'cncf', 'gnome', 'kde', 'openstack', 'osgeo', 'opensourcegeospatial',
    'softwareheritage', 'openknowledge', 'wikimedia', 'ourresearch', 'berkeley',
    'mit', 'stanford', 'stanforduniversity', 'audiopedia', 'audiopediafoundation',
    'opensourcesecurity', 'openssf', 'openjs', 'openjsfoundation', 'academysoftware',
    'academysoftwarefoundation', 'openmobility', 'openmobilityfoundation', 'osu',
    'opensource', 'fossi', 'fossifoundation', 'openwallet', 'openwalletfoundation',
    'verapdf', 'nomic', 'nomicfoundation', 'farama', 'faramafoundation', 'fintech',
    'fintechopensource', 'communityox', 'commonhaus'
}

# --- Data Collection ---
all_repo_data = []
unfiltered_batch = []
processed_repos = set()
language_index = 2 + prop_list.index('language')

def process_batch(batch, final_list, current_count, target):
    """Filters a batch by language and adds valid repos to the final list."""
    print(f"\nProcessing a batch of {len(batch)} repos...")
    newly_added = 0
    for repo_entry in batch:
        if current_count >= target:
            break
        language = repo_entry[language_index]
        if language in target_languages:
            final_list.append(repo_entry)
            current_count += 1
            newly_added += 1
    print(f"Added {newly_added} valid repositories. Total count: {current_count}/{target}")
    return current_count

# --- STEP 1: Read pre-filtered repositories directly from the CSV ---
print("Step 1: Reading pre-filtered repositories from the CSV file.")
try:
    df_existing = pd.read_csv(csv_file_path)
    
    # Define the columns the script expects to work with
    expected_columns = ['Owner account Name', 'Repo Name'] + prop_list
    
    # Ensure all necessary columns exist in the CSV
    if not all(col in df_existing.columns for col in expected_columns):
        raise ValueError("The CSV is missing required columns. Please ensure it matches the script's output format.")

    # Convert the DataFrame directly to the list format used by the script
    all_repo_data = df_existing[expected_columns].values.tolist()
    
    # Populate the processed_repos set to avoid fetching these repos again
    for index, row in df_existing.iterrows():
        repo_full_name = f"{row['Owner account Name']}/{row['Repo Name']}"
        processed_repos.add(repo_full_name)
        
except FileNotFoundError:
    print(f"Info: The file '{csv_file_path}' was not found. Starting a new collection from scratch.")
except Exception as e:
    print(f"An error occurred while reading the CSV in Step 1: {e}. Starting a new collection.")
    all_repo_data = []
    processed_repos = set()

current_repo_count = len(all_repo_data)
print(f"Successfully loaded {current_repo_count} repositories. Starting point: {current_repo_count}/{target_count}")

# --- STEP 2: Find and fetch new repos to reach the target ---
print(f"\nStep 2: Finding new repositories to append to the existing list.")
if current_repo_count < target_count:
    search_queries = [
    # Granular Star Ranges
    'stars:50..75',
    'stars:76..100',
    'stars:101..125',
    'stars:126..150',
    'stars:151..200',
    'stars:201..250',
    'stars:251..300',

    # Combining Stars and Forks
    'stars:50..100 forks:10..25',
    'stars:100..150 forks:25..50',
    'stars:150..200 forks:>50',

    # Different Date Ranges (for different years)
    'created:2024-01-01..2024-12-31 stars:50..100',
    'created:2023-01-01..2023-12-31 stars:75..125',
    'created:2022-01-01..2022-12-31 stars:100..150',

    # Recently Updated Repos
    'pushed:>2025-07-01 stars:20..80',  # Pushed in the last month
    'pushed:2025-01-01..2025-06-30 stars:50..100', # Pushed this year

    # Based on Size
    'size:5000..10000 stars:20..60',
    'size:10001..20000 stars:30..70',
    'size:>20000 stars:50..100'
]
    
    for query in search_queries:
        if current_repo_count >= target_count:
            break

        print(f"\nSearching with query: '{query}'")
        page = 1
        while current_repo_count < target_count:
            url = f'https://api.github.com/search/repositories?q={query}&sort=stars&order=desc&per_page=100&page={page}'
            
            try:
                response = requests.get(url, headers=headers)
                
                if response.status_code == 403 and 'rate limit exceeded' in response.text.lower():
                    reset_time = int(response.headers.get('X-RateLimit-Reset', 0))
                    wait_time = max(reset_time - time.time(), 0) + 5
                    print(f"Rate limit exceeded. Waiting {wait_time:.0f} seconds...")
                    time.sleep(wait_time)
                    continue

                if response.status_code != 200:
                    print(f"Error fetching search results ({response.status_code}): {response.text}")
                    break

                search_results = response.json()
                items = search_results.get('items', [])
                if not items:
                    print("No more results for this query.")
                    break

                for repo in items:
                    repo_full_name = repo['full_name']
                    owner_info = repo.get('owner', {})
                    
                    if repo_full_name in processed_repos or owner_info.get('type') != 'User' or owner_info.get('login', '').lower() in BLACKLISTED_OWNERS:
                        continue
                    
                    processed_repos.add(repo_full_name)
                    repo_properties = [repo.get(prop) if prop != 'license' else (repo.get('license') or {}).get('name') for prop in prop_list]
                    
                    owner, name = repo_full_name.split('/')
                    unfiltered_batch.append([owner, name] + repo_properties)
                    
                    if len(unfiltered_batch) >= 1000:
                        current_repo_count = process_batch(unfiltered_batch, all_repo_data, current_repo_count, target_count)
                        unfiltered_batch = []
                        if current_repo_count >= target_count:
                            break
                
                if current_repo_count >= target_count or len(items) < 100:
                    break

            except requests.exceptions.RequestException as e:
                print(f"A network error occurred: {e}. Waiting 30s.")
                time.sleep(30)
                continue

            page += 1
            time.sleep(5)

# --- FINAL BATCH PROCESSING ---
if unfiltered_batch and current_repo_count < target_count:
    current_repo_count = process_batch(unfiltered_batch, all_repo_data, current_repo_count, target_count)
    unfiltered_batch = []

print(f"\nTarget of {target_count} reached or all queries exhausted.")
print(f"Final count of collected repositories: {len(all_repo_data)}")

# --- STEP 3: Save the final DataFrame ---
if all_repo_data:
    columns = ['Owner account Name', 'Repo Name'] + prop_list
    df = pd.DataFrame(all_repo_data, columns=columns)
    df.to_csv(output_csv_path, index=False)
    print(f"Saved {len(df)} repository data to '{output_csv_path}'.")
else:
    print("No data collected. No CSV file was saved.")